# Build: Seq2Seq Date Formatter

**Module 9** — Sequence-to-Sequence Models

> From the [Zylo Learning Platform](https://socrapy-local.vercel.app) Sequence Models course.


## Context

This is a complete PyTorch implementation. Study the encoder-decoder architecture --- you'll recognize every piece from what we just learned.


## Setup

Install required packages (skip if already installed):


In [ ]:
!pip install torch -q


## Code

Run each cell in order:


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

### Step 2: --- Data ---


In [ ]:
def generate_date_pairs(n=1000):
    """Generate (human_date, iso_date) pairs."""
    months = ['January','February','March','April','May','June',
              'July','August','September','October','November','December']
    pairs = []
    for _ in range(n):
        m = random.randint(1, 12)
        d = random.randint(1, 28)
        y = random.randint(1990, 2025)
        human = f'{months[m-1]} {d}, {y}'   # 'March 15, 2021'
        iso = f'{y}-{m:02d}-{d:02d}'        # '2021-03-15'
        pairs.append((human, iso))
    return pairs

### Step 3: Character-level vocabulary


In [ ]:
def build_vocab(pairs):
    chars = set()
    for src, tgt in pairs:
        chars.update(src)
        chars.update(tgt)
    chars = sorted(chars)
    stoi = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2}
    for i, c in enumerate(chars):
        stoi[c] = i + 3
    itos = {v: k for k, v in stoi.items()}
    return stoi, itos

def encode_seq(s, stoi, max_len):
    return [stoi[c] for c in s] + [stoi['<EOS>']] + [stoi['<PAD>']] * (max_len - len(s))

### Step 4: --- Model ---


In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
    
    def forward(self, x):
        embedded = self.embed(x)            # [B, T, E]
        outputs, (h, c) = self.lstm(embedded)  # h = context vector
        return h, c

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, h, c):
        embedded = self.embed(x.unsqueeze(1))  # [B, 1, E]
        output, (h, c) = self.lstm(embedded, (h, c))
        pred = self.fc(output.squeeze(1))      # [B, vocab_size]
        return pred, h, c

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    
    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        vocab_size = self.decoder.fc.out_features
        
        outputs = torch.zeros(batch_size, tgt_len, vocab_size).to(self.device)
        h, c = self.encoder(src)
        
        # First input is <SOS>
        inp = tgt[:, 0]  # <SOS> token
        
        for t in range(1, tgt_len):
            pred, h, c = self.decoder(inp, h, c)
            outputs[:, t] = pred
            
            # Teacher forcing
            if random.random() < teacher_forcing_ratio:
                inp = tgt[:, t]     # ground truth
            else:
                inp = pred.argmax(1)  # model's prediction
        
        return outputs

### Step 5: --- Training ---


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pairs = generate_date_pairs(5000)
stoi, itos = build_vocab(pairs)
vocab_size = len(stoi)

EMBED_DIM = 32
HIDDEN_DIM = 64

encoder = Encoder(vocab_size, EMBED_DIM, HIDDEN_DIM)
decoder = Decoder(vocab_size, EMBED_DIM, HIDDEN_DIM)
model = Seq2Seq(encoder, decoder, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss(ignore_index=stoi['<PAD>'])

print(f'Vocab size: {vocab_size}')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Example: "{pairs[0][0]}" -> "{pairs[0][1]}"')

## Key Takeaway

The date formatter is deceptively simple, but it contains every component of a real seq2seq system: character-level vocabulary, encoder compression, decoder autoregression, teacher forcing, and `<SOS>`/`<EOS>` tokens. If you understand this, you understand machine translation at the architectural level.

---

*Generated from the [Zylo Sequence Models Course](https://socrapy-local.vercel.app). Continue learning at the platform for interactive exercises, quizzes, and AI tutoring.*
